# Notebook 04 — Studi Kasus 2: Regresi

**Materi persiapan Ekshibisi Kecerdasan Artifisial · LKS Dikmen Nasional 2026**
Sasaran: siswa SMKN Bali Mandara, Program Keahlian Teknik Komputer dan Jaringan (TKJ)

Prasyarat: **Notebook 01, 02, dan 03** sudah selesai.

---

## Regresi versus klasifikasi

| | Klasifikasi (Notebook 03) | Regresi (notebook ini) |
|---|---|---|
| Keluaran | satu label dari pilihan terbatas | angka bebas |
| Contoh | organik / anorganik / B3 | 12,4 kWh |
| Contoh lain | hoaks / bukan hoaks | suhu besok 31,2 °C |
| Metrik | accuracy, precision, recall, F1 | MAE, RMSE, R² |
| Pemeriksaan | confusion matrix | plot prediksi vs kenyataan, residual |

**Alur kerjanya sama persis** dengan Notebook 03: muat, EDA, bagi, baseline,
pipeline, bandingkan, setel, uji sekali, simpan. Yang berganti hanya metrik dan
cara mendiagnosis kesalahan. Kalau Notebook 03 sudah dipahami, notebook ini akan
terasa familiar.

## Kasus yang dikerjakan

Menebak nilai tengah harga rumah di sebuah blok sensus di California dari
delapan keterangan wilayah itu.

Estimasi waktu: 120–150 menit.

## Bagian 0 — Persiapan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict,
                                     KFold, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import sklearn, joblib

SEED = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 130)

print("scikit-learn:", sklearn.__version__)

---

## Bagian 1 — Memuat dataset

### Dataset: California Housing

| Keterangan | Isi |
|---|---|
| Nama | California Housing dataset |
| Sumber asli | Sensus Amerika Serikat 1990, per *block group* |
| Rujukan | Pace RK & Barry R (1997), *Sparse Spatial Autoregressions*, Statistics & Probability Letters 33(3):291–297 |
| Disediakan oleh | scikit-learn, dari StatLib |
| Lisensi | domain publik |
| Ukuran | 20.640 baris, 8 fitur, 1 target |
| Cara memuat | `fetch_california_housing()` — mengunduh sekali, lalu tersimpan |

**Satu baris = satu blok sensus**, bukan satu rumah. Rata-rata satu blok berisi
600–3.000 penduduk. Ini penting: seluruh angka pada dataset ini adalah **rata-rata
wilayah**, bukan sifat sebuah rumah. Salah paham soal ini membuat kesimpulanmu
salah, seberapa pun bagus modelnya.

### Satu hal tentang dataset harga rumah yang perlu kamu tahu

Dahulu scikit-learn juga menyediakan "Boston Housing". Dataset itu **dihapus**
karena salah satu kolomnya dibuat berdasarkan komposisi ras penduduk — sebuah
variabel yang tidak layak dipakai untuk memprediksi harga, dan yang membuat model
apa pun yang dilatih darinya mewarisi asumsi rasis.

Pelajarannya bukan soal Boston. Pelajarannya: **periksa arti setiap kolom sebelum
memakainya.** Sebuah kolom bisa menaikkan akurasi dan tetap tidak layak dipakai.
Pertanyaan "apakah fitur ini pantas dipakai?" adalah pertanyaan manusia, dan tidak
ada algoritma yang bisa menjawabnya untukmu.

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame

print(f"{df.shape[0]:,} baris, {df.shape[1]} kolom")
print()
print("Nilai hilang:", int(df.isna().sum().sum()))
print("Baris ganda :", int(df.duplicated().sum()))
df.head()

> **Kalau sel di atas gagal mengunduh** (Colab tanpa internet):
> jalankan `df = pd.read_csv("<tautan salinanmu>")`, atau unduh dulu berkasnya
> lewat peramban dan unggah ke Colab. `fetch_california_housing` menyimpan
> hasil unduhan, jadi pemanggilan kedua tidak mengunduh lagi.

### Arti setiap kolom

| Kolom | Arti | Satuan |
|---|---|---|
| `MedInc` | pendapatan tengah rumah tangga di blok itu | puluhan ribu USD |
| `HouseAge` | usia tengah bangunan | tahun |
| `AveRooms` | rata-rata jumlah ruangan per rumah tangga | ruangan |
| `AveBedrms` | rata-rata jumlah kamar tidur per rumah tangga | kamar |
| `Population` | jumlah penduduk blok | orang |
| `AveOccup` | rata-rata penghuni per rumah tangga | orang |
| `Latitude` | lintang | derajat |
| `Longitude` | bujur | derajat |
| **`MedHouseVal`** | **target:** nilai tengah harga rumah | **ratusan ribu USD** |

Satuan target: `2.5` berarti 250.000 USD. Angka ini akan muncul di setiap metrik
nanti, jadi hafalkan satuannya sekarang — kalau tidak, MAE `0,3` tidak berarti
apa-apa bagimu.

In [ ]:
df.describe().round(2)

### Dua temuan dari `describe` yang wajib ditindaklanjuti

**1. `AveRooms` maksimumnya 141,9 dan `AveOccup` maksimumnya 1.243.**
Tidak ada rumah dengan rata-rata 141 ruangan, dan tidak ada 1.243 orang per
rumah tangga. Nilai-nilai ini muncul karena blok dengan penduduk sangat sedikit
membuat pembaginya kecil. Bukan salah ketik, tapi tetap perlu diperlakukan
hati-hati.

**2. `MedHouseVal` maksimumnya tepat 5,00.**
Ini bukan kebetulan. Nilai di atas 500.000 USD **dipotong** menjadi 5,00001 pada
data aslinya. Data seperti ini disebut **tersensor** (*censored*).

Akibatnya langsung: model tidak akan pernah bisa menebak rumah yang harganya
lebih dari 500.000 USD, karena informasi itu tidak ada di datanya. Berapa banyak
baris yang terkena? Mari dihitung — jangan diperkirakan.

In [ ]:
tersensor = (df["MedHouseVal"] >= 5.0).sum()
print(f"Baris dengan target dipotong di 5.00: {tersensor:,} dari {len(df):,} "
      f"({tersensor / len(df) * 100:.1f}%)")

aveocc_ekstrem = (df["AveOccup"] > 10).sum()
averoom_ekstrem = (df["AveRooms"] > 15).sum()
print(f"AveOccup > 10 orang/rumah tangga     : {aveocc_ekstrem:,} baris")
print(f"AveRooms > 15 ruangan/rumah tangga   : {averoom_ekstrem:,} baris")

4,8% data tersensor. Itu cukup besar untuk harus disebutkan dalam laporan, tapi
tidak cukup besar untuk membuat dataset tidak berguna. Kita **pertahankan** dan
**catat batasannya**: model ini tidak bisa dipakai untuk rumah di atas 500.000 USD.

Menghapusnya juga pilihan yang sah, tapi berarti membuang 4,8% data dan membuat
model buta terhadap wilayah termahal. Keputusan ini harus disebutkan, bukan
diambil diam-diam.

---

## Bagian 2 — EDA untuk regresi

Bedanya dengan EDA klasifikasi: di sini **target berupa angka**, jadi yang dilihat
adalah sebarannya dan hubungannya dengan setiap fitur.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(df["MedHouseVal"], bins=50, color="#4C72B0", edgecolor="white")
ax[0].axvline(5.0, color="#C44E52", linestyle="--", linewidth=2, label="batas sensor 5.00")
ax[0].set_xlabel("MedHouseVal (ratusan ribu USD)")
ax[0].set_ylabel("jumlah blok")
ax[0].set_title("Sebaran target")
ax[0].legend()

sns.boxplot(x=df["MedHouseVal"], ax=ax[1], color="#4C72B0")
ax[1].set_xlabel("MedHouseVal (ratusan ribu USD)")
ax[1].set_title("Boxplot target")

plt.tight_layout()
plt.show()

print(f"rata-rata {df['MedHouseVal'].mean():.3f} | median {df['MedHouseVal'].median():.3f}")

Tiang tinggi di ujung kanan, tepat di 5,00, adalah penumpukan akibat sensor.
Temuan ini hanya terlihat kalau histogramnya benar-benar dilihat — `describe()`
saja hanya menampilkan `max = 5.00` tanpa memberi tahu ada 992 baris di sana.

Inilah alasan EDA tidak bisa dilewati: angka ringkasan menyembunyikan bentuk.

In [ ]:
# Korelasi setiap fitur dengan target, diurutkan
korelasi = df.corr(numeric_only=True)["MedHouseVal"].drop("MedHouseVal").sort_values()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

warna = ["#C44E52" if v < 0 else "#4C72B0" for v in korelasi]
ax[0].barh(korelasi.index, korelasi.values, color=warna)
ax[0].axvline(0, color="black", linewidth=0.8)
ax[0].set_xlabel("korelasi dengan MedHouseVal")
ax[0].set_title("Korelasi fitur terhadap target")

sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax[1], cbar=False, annot_kws={"size": 7})
ax[1].set_title("Korelasi antar semua kolom")

plt.tight_layout()
plt.show()

print(korelasi.round(3))

`MedInc` (pendapatan) berkorelasi 0,69 dengan harga — jauh lebih kuat daripada
kolom lain. Sudah bisa diduga dari sekarang bahwa kolom ini akan menjadi fitur
terpenting.

Perhatikan juga `AveRooms` dan `AveBedrms` berkorelasi 0,85 satu sama lain.
Keduanya mengukur hal yang mirip. Ini akan kita manfaatkan di Bagian 8.

In [ ]:
# Scatter plot fitur terkuat terhadap target
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

contoh = df.sample(3000, random_state=SEED)   # disampel supaya grafiknya terbaca

ax[0].scatter(contoh["MedInc"], contoh["MedHouseVal"], alpha=0.25, s=12, color="#4C72B0")
ax[0].axhline(5.0, color="#C44E52", linestyle="--", label="batas sensor")
ax[0].set_xlabel("MedInc (puluhan ribu USD)")
ax[0].set_ylabel("MedHouseVal")
ax[0].set_title("Pendapatan vs harga rumah")
ax[0].legend()

ax[1].scatter(contoh["HouseAge"], contoh["MedHouseVal"], alpha=0.25, s=12, color="#55A868")
ax[1].set_xlabel("HouseAge (tahun)")
ax[1].set_ylabel("MedHouseVal")
ax[1].set_title("Usia bangunan vs harga rumah")

plt.tight_layout()
plt.show()

Grafik kiri menunjukkan hubungan naik yang jelas, **dan** garis lurus mendatar di
atas — penumpukan akibat sensor terlihat langsung di sini.

Grafik kanan hampir tidak menunjukkan pola. `HouseAge` sendirian tidak banyak
berguna, walaupun bisa tetap berguna bila digabungkan dengan kolom lain.

In [ ]:
# Sebaran geografis: data ini punya lintang dan bujur, jadi bisa dipetakan
fig, ax = plt.subplots(figsize=(7.5, 7))
sc = ax.scatter(df["Longitude"], df["Latitude"], c=df["MedHouseVal"],
                cmap="viridis", s=5, alpha=0.5)
plt.colorbar(sc, ax=ax, label="MedHouseVal (ratusan ribu USD)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Harga rumah di California — bentuk peta terlihat dengan sendirinya")
plt.tight_layout()
plt.show()

Peta California terbentuk sendiri dari titik-titiknya, dan daerah mahal
mengumpul di dua tempat: sekitar San Francisco (kira-kira 37,8 LU / 122,4 BB) dan
Los Angeles (34,1 LU / 118,2 BB), serta di sepanjang pantai.

Ini temuan EDA yang paling berharga di notebook ini, dan akan langsung kita pakai
di Bagian 8 untuk membuat fitur baru. Tanpa memandangi peta ini, gagasan itu tidak
akan muncul.

---

## Bagian 3 — Bagi data dan buat baseline

In [ ]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

print(f"Data latih : {len(X_latih):,} baris")
print(f"Data uji   : {len(X_uji):,} baris")
print()
print(f"rata-rata target latih : {y_latih.mean():.4f}")
print(f"rata-rata target uji   : {y_uji.mean():.4f}   <- mirip, pembagiannya wajar")

Tidak ada `stratify` di sini — argumen itu hanya untuk label kategori. Pada
regresi, pembagian acak biasa sudah cukup asalkan datanya banyak. Kalau datanya
sedikit dan sebaran targetnya sangat miring, bisa dipakai `stratify` atas target
yang sudah dikelompokkan dengan `pd.qcut`.

**Aturan Notebook 03 tetap berlaku:** `X_uji` dan `y_uji` tidak disentuh sampai
Bagian 9.

In [ ]:
# Baseline regresi: selalu menjawab rata-rata target data latih
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_latih, y_latih)

tebakan_base = baseline.predict(X_latih)
mae_base = mean_absolute_error(y_latih, tebakan_base)
rmse_base = np.sqrt(mean_squared_error(y_latih, tebakan_base))
r2_base = r2_score(y_latih, tebakan_base)

print(f"Selalu menjawab      : {baseline.constant_[0][0]:.4f}")
print(f"MAE  baseline        : {mae_base:.4f}  (= {mae_base * 100_000:,.0f} USD)")
print(f"RMSE baseline        : {rmse_base:.4f}")
print(f"R2   baseline        : {r2_base:.4f}")

`R² = 0` untuk baseline, dan itu memang definisinya: R² mengukur perbaikan
terhadap "selalu menjawab rata-rata". Model yang lebih buruk dari baseline akan
mendapat **R² negatif** — hal yang sangat mungkin terjadi, dan kalau terjadi
berarti ada yang salah.

---

## Bagian 4 — Tiga metrik regresi

| Metrik | Arti | Satuan | Ciri |
|---|---|---|---|
| **MAE** | rata-rata besarnya salah tebak | sama dengan target | mudah dijelaskan |
| **RMSE** | akar rata-rata salah tebak dikuadratkan | sama dengan target | menghukum salah besar |
| **R²** | berapa bagian keragaman data yang terjelaskan | tanpa satuan, 0–1 | mudah dibandingkan |

### MAE atau RMSE — bedanya nyata

Keduanya bersatuan sama, tapi menjawab pertanyaan berbeda. RMSE mengkuadratkan
kesalahan sebelum dirata-ratakan, sehingga satu kesalahan besar berpengaruh jauh
lebih banyak daripada beberapa kesalahan kecil.

Contoh angkanya lebih meyakinkan daripada penjelasannya:

In [ ]:
asli = np.array([2.0, 2.0, 2.0, 2.0, 2.0])

merata = np.array([2.2, 1.8, 2.2, 1.8, 2.2])       # semua salah sedikit
terpusat = np.array([2.0, 2.0, 2.0, 2.0, 3.0])     # empat tepat, satu salah besar

for nama, tebak in [("salah merata", merata), ("satu salah besar", terpusat)]:
    mae = mean_absolute_error(asli, tebak)
    rmse = np.sqrt(mean_squared_error(asli, tebak))
    print(f"{nama:<18} MAE {mae:.3f} | RMSE {rmse:.3f} | RMSE/MAE {rmse / mae:.2f}")

MAE-nya nyaris sama, tapi RMSE kasus kedua jauh lebih besar.

**Cara memilih — ini pertanyaan tentang akibat di dunia nyata, bukan tentang rumus:**

- Kalau satu kesalahan besar jauh lebih merugikan daripada beberapa kesalahan
  kecil → pakai **RMSE**. Contoh: memperkirakan kebutuhan daya listrik. Salah
  besar sekali berarti listrik mati.
- Kalau semua kesalahan setara → pakai **MAE**. Lebih mudah dijelaskan juga:
  "rata-rata salah 30.000 USD" langsung dimengerti siapa pun.

**Laporkan keduanya.** Perbandingan RMSE/MAE sendiri sudah informatif: kalau jauh
di atas 1, berarti ada sedikit kesalahan yang sangat besar — dan itu petunjuk ke
mana harus melihat.

---

## Bagian 5 — Melatih dan membandingkan model

Regresi memakai kelas yang berbeda dari klasifikasi, tapi namanya bersesuaian:

| Klasifikasi (Notebook 03) | Regresi (di sini) |
|---|---|
| `LogisticRegression` | `LinearRegression` / `Ridge` |
| `DecisionTreeClassifier` | `DecisionTreeRegressor` |
| `RandomForestClassifier` | `RandomForestRegressor` |
| `DummyClassifier` | `DummyRegressor` |

Semua kolom di sini berupa angka, jadi `ColumnTransformer` tidak diperlukan —
cukup `StandardScaler` untuk semuanya.

`scoring="neg_root_mean_squared_error"` memakai tanda negatif karena
scikit-learn selalu menganggap "lebih besar lebih baik". Jadi nilainya perlu
dikalikan -1 agar terbaca sebagai RMSE biasa.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

kandidat = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Decision Tree (d=8)": DecisionTreeRegressor(max_depth=8, random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=SEED),
}

hasil = []
for nama, algoritma in kandidat.items():
    pipa = Pipeline([("skala", StandardScaler()), ("model", algoritma)])
    rmse = -cross_val_score(pipa, X_latih, y_latih, cv=cv,
                            scoring="neg_root_mean_squared_error")
    mae = -cross_val_score(pipa, X_latih, y_latih, cv=cv,
                           scoring="neg_mean_absolute_error")
    r2 = cross_val_score(pipa, X_latih, y_latih, cv=cv, scoring="r2")
    hasil.append({"model": nama, "RMSE": rmse.mean(), "RMSE_sd": rmse.std(),
                  "MAE": mae.mean(), "R2": r2.mean()})
    print(f"{nama:<24} selesai")

tabel = pd.DataFrame(hasil).sort_values("RMSE").reset_index(drop=True)
print()
print(f"{'BASELINE':<24} RMSE {rmse_base:.4f} | MAE {mae_base:.4f} | R2 0.0000")
print()
print(tabel.round(4).to_string(index=False))

Bacaan dari tabel itu:

1. **Semua model mengalahkan baseline** (RMSE 1,14). Bagus — berarti fiturnya
   memang mengandung informasi.
2. **Model linear tertinggal jauh** dari model berbasis pohon. Artinya hubungan
   antara fitur dan harga **tidak berupa garis lurus** — yang sudah terlihat pada
   scatter plot dan peta di Bagian 2.
3. **Selisih antar model besar** dibandingkan simpangannya (`RMSE_sd` sekitar
   0,01). Di sini, berbeda dengan Notebook 03, perbandingan modelnya benar-benar
   berarti.
4. Boosting sedikit mengalahkan Random Forest dan jauh lebih cepat dilatih.

Bandingkan dengan Notebook 03, di mana semua model praktis sama baik. Itu bukan
karena algoritmanya berbeda, tapi karena masalahnya berbeda: pinguin mudah
dipisahkan, harga rumah tidak.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].barh(tabel["model"], tabel["RMSE"], xerr=tabel["RMSE_sd"],
           color="#4C72B0", capsize=4)
ax[0].axvline(rmse_base, color="#C44E52", linestyle="--", label=f"baseline {rmse_base:.2f}")
ax[0].set_xlabel("RMSE (lebih kecil lebih baik)")
ax[0].set_title("RMSE cross-validation")
ax[0].legend()
ax[0].invert_yaxis()

ax[1].barh(tabel["model"], tabel["R2"], color="#55A868")
ax[1].set_xlabel("R2 (lebih besar lebih baik)")
ax[1].set_xlim(0, 1)
ax[1].set_title("R2 cross-validation")
ax[1].invert_yaxis()

plt.tight_layout()
plt.show()

---

## Bagian 6 — Mendiagnosis kesalahan model

Pada klasifikasi, alat diagnosisnya confusion matrix. Pada regresi, tiga grafik
berikut yang dipakai. Semuanya dihitung dengan `cross_val_predict` di data latih
— data uji masih tertutup.

In [ ]:
model_kerja = Pipeline([
    ("skala", StandardScaler()),
    ("model", HistGradientBoostingRegressor(random_state=SEED)),
])

tebakan_cv = cross_val_predict(model_kerja, X_latih, y_latih, cv=cv)
sisa = y_latih - tebakan_cv          # residual: kenyataan - tebakan

print(f"MAE  : {mean_absolute_error(y_latih, tebakan_cv):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_latih, tebakan_cv)):.4f}")
print(f"R2   : {r2_score(y_latih, tebakan_cv):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

# 1 — Prediksi vs kenyataan. Titik ideal berada di garis diagonal.
ax[0].scatter(y_latih, tebakan_cv, alpha=0.15, s=6, color="#4C72B0")
ax[0].plot([0, 5.2], [0, 5.2], color="#C44E52", linestyle="--", linewidth=2)
ax[0].set_xlabel("kenyataan")
ax[0].set_ylabel("tebakan model")
ax[0].set_title("Prediksi vs kenyataan")
ax[0].set_xlim(0, 5.3)
ax[0].set_ylim(0, 5.3)

# 2 — Residual vs tebakan. Idealnya menyebar rata di sekitar nol.
ax[1].scatter(tebakan_cv, sisa, alpha=0.15, s=6, color="#55A868")
ax[1].axhline(0, color="#C44E52", linestyle="--", linewidth=2)
ax[1].set_xlabel("tebakan model")
ax[1].set_ylabel("residual (kenyataan - tebakan)")
ax[1].set_title("Residual")

# 3 — Sebaran residual. Idealnya simetris dan berpusat di nol.
ax[2].hist(sisa, bins=60, color="#8172B2", edgecolor="white")
ax[2].axvline(0, color="#C44E52", linestyle="--", linewidth=2)
ax[2].set_xlabel("residual")
ax[2].set_ylabel("jumlah blok")
ax[2].set_title("Sebaran residual")

plt.tight_layout()
plt.show()

print(f"rata-rata residual : {sisa.mean():+.4f}  (idealnya mendekati 0)")
print(f"residual terbesar  : {sisa.max():+.4f}")
print(f"residual terkecil  : {sisa.min():+.4f}")

### Cara membaca ketiga grafik itu

**Grafik 1 — prediksi vs kenyataan.** Titik ideal menempel di garis diagonal.
Perhatikan kolom titik yang tegak lurus di `kenyataan = 5,0`: model menebak
bermacam-macam nilai untuk blok yang sebenarnya semua bernilai 5,0. Itu bukan
kesalahan model — itu akibat sensor yang kita temukan di Bagian 1. Model tidak
bisa membedakan rumah 500 ribu dari rumah 900 ribu karena datanya memang tidak
membedakan keduanya.

**Grafik 2 — residual.** Idealnya sebaran titik rata di atas dan di bawah garis
nol tanpa pola. Di sini ada pola berbentuk kipas: makin tinggi tebakannya, makin
lebar sebaran kesalahannya. Artinya model **kurang andal pada rumah mahal**. Ini
harus disebutkan dalam laporan, dan MAE tunggal tidak menunjukkannya.

**Grafik 3 — sebaran residual.** Kalau puncaknya bergeser dari nol, berarti model
punya kecenderungan menebak terlalu tinggi atau terlalu rendah secara sistematis.
Di sini cukup berpusat, tapi ekornya panjang — ada sedikit blok yang salah tebak
sangat besar.

Ketiga grafik ini menjawab pertanyaan berbeda. Kalau hanya sempat membuat satu,
pilih grafik 2.

In [ ]:
# Sepuluh kesalahan terburuk — selalu lihat ini, sering langsung ketemu pola
periksa = X_latih.copy()
periksa["kenyataan"] = y_latih
periksa["tebakan"] = tebakan_cv
periksa["selisih"] = sisa
periksa["selisih_mutlak"] = sisa.abs()

terburuk = periksa.nlargest(10, "selisih_mutlak")[
    ["MedInc", "HouseAge", "AveRooms", "AveOccup", "Population",
     "kenyataan", "tebakan", "selisih"]]
print("Sepuluh kesalahan terbesar:")
print(terburuk.round(2).to_string())
print()
print(f"Dari 10 kesalahan terbesar, yang kenyataannya tersensor di 5.00: "
      f"{(terburuk['kenyataan'] >= 5.0).sum()} dari 10")

Sebagian besar kesalahan terbesar terjadi pada blok yang targetnya tersensor.
Artinya sebagian dari "kesalahan" model itu **tidak bisa diperbaiki** dengan
algoritma apa pun — batasnya ada di datanya.

Mengetahui hal ini menyelamatkan waktu: tidak ada gunanya menyetel model untuk
memperbaiki kesalahan yang penyebabnya di data. Dan ini jawaban yang bagus untuk
pertanyaan juri "apa keterbatasan modelmu?".

---

## Bagian 7 — Fitur mana yang berpengaruh

In [ ]:
model_kerja.fit(X_latih, y_latih)

# Disampel 3000 baris supaya cepat; permutation importance mahal pada data besar
sampel = X_latih.sample(3000, random_state=SEED)
imp = permutation_importance(model_kerja, sampel, y_latih.loc[sampel.index],
                            n_repeats=8, random_state=SEED,
                            scoring="neg_root_mean_squared_error")

tabel_imp = (pd.DataFrame({
    "fitur": X_latih.columns,
    "kenaikan_rmse": imp.importances_mean,
    "simpangan": imp.importances_std,
}).sort_values("kenaikan_rmse", ascending=False).reset_index(drop=True))

print(tabel_imp.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(tabel_imp["fitur"], tabel_imp["kenaikan_rmse"],
        xerr=tabel_imp["simpangan"], color="#55A868", capsize=3)
ax.set_xlabel("kenaikan RMSE bila kolom ini diacak")
ax.set_title("Permutation importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Hasil ini bertentangan dengan dugaan dari korelasi — dan itu menarik

Di Bagian 2, `MedInc` punya korelasi tertinggi dengan target (0,69), sementara
`Latitude` hanya sekitar -0,14. Tetapi menurut permutation importance, **lintang
dan bujur justru lebih berpengaruh daripada pendapatan.**

Keduanya tidak bertentangan; keduanya mengukur hal yang berbeda:

- **Korelasi** hanya mengukur hubungan **garis lurus** antara satu kolom dan
  target. Pengaruh lokasi terhadap harga sama sekali bukan garis lurus — harga
  tidak naik terus ke utara, tapi memuncak di dua tempat. Korelasi tidak bisa
  melihat bentuk seperti itu, jadi nilainya kecil.
- **Permutation importance** mengukur seberapa besar skor **model** turun bila
  kolom itu dirusak. Model berbasis pohon sanggup memanfaatkan pola tidak lurus,
  jadi ia memang sangat bergantung pada koordinat.

Pelajarannya: **jangan memilih atau membuang fitur hanya berdasarkan korelasi.**
Korelasi rendah tidak berarti tidak berguna. Kalau di Bagian 2 kita membuang
`Latitude` karena korelasinya kecil, model ini akan jauh lebih buruk.

Kenyataan bahwa lokasi begitu dominan memberi gagasan yang akan kita uji sekarang
— dan juga menjadi risiko yang kita bahas di Bagian 12.

---

## Bagian 8 — Rekayasa fitur: satu gagasan mengalahkan semua penyetelan

Model tahu lintang dan bujur, tapi ia tidak tahu bahwa San Francisco dan Los
Angeles itu penting. Ia harus menemukannya sendiri dari angka koordinat — kerja
yang sulit untuk model berbasis pohon, karena membutuhkan banyak sekali
pemotongan untuk mendekati bentuk jarak.

Kita bantu dengan menghitungkannya: **jarak ke kota besar terdekat**.

Ini yang disebut **pengetahuan bidang** (*domain knowledge*) — dan sering menjadi
satu-satunya hal yang benar-benar membedakan hasil. Tidak ada algoritma yang bisa
menebak bahwa dua titik koordinat tertentu adalah pusat ekonomi California.
Manusia yang tahu itu.

Dua fitur lain kita tambahkan berdasarkan temuan Bagian 2: `AveRooms` dan
`AveBedrms` berkorelasi 0,85, jadi rasio keduanya mungkin lebih informatif
daripada masing-masing.

In [ ]:
LA = (34.05, -118.24)      # Los Angeles
SF = (37.77, -122.42)      # San Francisco


def tambah_fitur(d):
    'Menambahkan tiga kolom hasil rekayasa. Bekerja untuk data latih maupun data baru.'
    d = d.copy()
    d["kamar_per_orang"] = d["AveRooms"] / d["AveOccup"]
    d["rasio_kamar_tidur"] = d["AveBedrms"] / d["AveRooms"]
    jarak_la = np.hypot(d["Latitude"] - LA[0], d["Longitude"] - LA[1])
    jarak_sf = np.hypot(d["Latitude"] - SF[0], d["Longitude"] - SF[1])
    d["jarak_kota"] = np.minimum(jarak_la, jarak_sf)
    return d


X_latih_plus = tambah_fitur(X_latih)
X_uji_plus = tambah_fitur(X_uji)      # rumus yang sama, TIDAK dihitung dari data uji

print("Kolom sebelum :", X_latih.shape[1])
print("Kolom sesudah :", X_latih_plus.shape[1])
print()
X_latih_plus[["Latitude", "Longitude", "jarak_kota", "kamar_per_orang",
              "rasio_kamar_tidur"]].head().round(3)

Perhatikan satu hal penting: ketiga fitur ini dihitung **hanya dari kolom pada
baris itu sendiri**. Tidak ada rata-rata seluruh data yang terlibat. Karena itu
aman dihitung di luar `Pipeline`.

Kalau fiturnya melibatkan statistik seluruh dataset — misalnya "pendapatan dibagi
rata-rata pendapatan semua blok" — itu **harus** di dalam `Pipeline`, kalau tidak
informasi data uji bocor ke data latih.

Sekarang diukur: apakah ketiga fitur ini benar-benar membantu?

In [ ]:
rmse_tanpa = -cross_val_score(model_kerja, X_latih, y_latih, cv=cv,
                              scoring="neg_root_mean_squared_error")
rmse_dengan = -cross_val_score(model_kerja, X_latih_plus, y_latih, cv=cv,
                               scoring="neg_root_mean_squared_error")

print(f"RMSE fitur asli      : {rmse_tanpa.mean():.4f} +/- {rmse_tanpa.std():.4f}")
print(f"RMSE + 3 fitur baru  : {rmse_dengan.mean():.4f} +/- {rmse_dengan.std():.4f}")
print(f"Perbaikan            : {rmse_tanpa.mean() - rmse_dengan.mean():+.4f} "
      f"({(rmse_tanpa.mean() - rmse_dengan.mean()) / rmse_tanpa.mean() * 100:+.1f}%)")
print()
print(f"Simpangan CV         : {rmse_dengan.std():.4f}")
print("-> Perbaikannya lebih besar daripada simpangan, jadi bukan kebetulan.")

Perbaikan nyata, dan didapat dari tiga baris kode — bukan dari algoritma yang
lebih rumit. Lebih penting lagi: **perbaikannya lebih besar daripada simpangan
cross-validation**, jadi ini bukan keberuntungan pembagian data.

Inilah urutan prioritas yang benar dalam memperbaiki model:

1. **Data lebih banyak atau lebih bersih** — dampak paling besar.
2. **Fitur yang lebih baik** — dampak besar, biaya rendah. (Bagian ini.)
3. **Algoritma yang berbeda** — dampak sedang.
4. **Penyetelan hyperparameter** — dampak paling kecil, sering hanya 1–2%.

Peserta lomba sering mengerjakannya dari bawah ke atas. Kerjakan dari atas.

---

## Bagian 9 — Penyetelan

In [ ]:
ruang = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_leaf_nodes": [31, 63],
    "model__max_iter": [200, 400],
}

pencarian = GridSearchCV(
    Pipeline([("skala", StandardScaler()),
              ("model", HistGradientBoostingRegressor(random_state=SEED))]),
    param_grid=ruang,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
)

pencarian.fit(X_latih_plus, y_latih)

print(f"Kombinasi dicoba : {len(pencarian.cv_results_['params'])}")
print(f"RMSE CV terbaik  : {-pencarian.best_score_:.4f}")
print(f"Sebelum disetel  : {rmse_dengan.mean():.4f}")
print(f"Perbaikan tuning : {rmse_dengan.mean() - (-pencarian.best_score_):+.4f}")
print()
for k, v in pencarian.best_params_.items():
    print(f"   {k} = {v}")

In [ ]:
ringkas = (pd.DataFrame(pencarian.cv_results_)
           [["param_model__learning_rate", "param_model__max_leaf_nodes",
             "param_model__max_iter", "mean_train_score", "mean_test_score",
             "std_test_score"]]
           .assign(mean_train_score=lambda d: -d["mean_train_score"],
                   mean_test_score=lambda d: -d["mean_test_score"])
           .sort_values("mean_test_score")
           .head(5).round(4))
ringkas.columns = ["learning_rate", "max_leaf_nodes", "max_iter",
                   "RMSE_latih", "RMSE_cv", "simpangan"]
print(ringkas.to_string(index=False))

Bandingkan dua perbaikan yang kita dapat:

- **Rekayasa fitur** (Bagian 8): tiga baris kode.
- **Penyetelan** (bagian ini): delapan kombinasi, jauh lebih lama dijalankan.

Perhatikan juga kolom `RMSE_latih` jauh lebih kecil daripada `RMSE_cv` — model
ini tetap menghafal sebagian data latih. Itu wajar untuk boosting, dan yang
penting adalah `RMSE_cv` tidak memburuk.

---

## Bagian 10 — Pengujian akhir

Sekali saja.

In [ ]:
model_akhir = pencarian.best_estimator_
tebakan_uji = model_akhir.predict(X_uji_plus)

mae_uji = mean_absolute_error(y_uji, tebakan_uji)
rmse_uji = np.sqrt(mean_squared_error(y_uji, tebakan_uji))
r2_uji = r2_score(y_uji, tebakan_uji)

print("=" * 62)
print("HASIL AKHIR PADA DATA UJI")
print("=" * 62)
print(f"Jumlah data uji : {len(y_uji):,} baris")
print()
print(f"{'metrik':<8}{'baseline':>12}{'model':>12}{'perbaikan':>14}")
print("-" * 46)
print(f"{'MAE':<8}{mae_base:>12.4f}{mae_uji:>12.4f}{(1 - mae_uji / mae_base) * 100:>13.1f}%")
print(f"{'RMSE':<8}{rmse_base:>12.4f}{rmse_uji:>12.4f}{(1 - rmse_uji / rmse_base) * 100:>13.1f}%")
print(f"{'R2':<8}{0.0:>12.4f}{r2_uji:>12.4f}{'':>14}")
print("-" * 46)
print()
print(f"RMSE CV (data latih) : {-pencarian.best_score_:.4f}")
print(f"RMSE data uji        : {rmse_uji:.4f}   <- mirip, berarti tidak overfitting")
print()
print(f"Dalam satuan USD: rata-rata salah tebak sekitar {mae_uji * 100_000:,.0f} USD")
print("=" * 62)

RMSE data uji mirip dengan RMSE cross-validation. Itu pemeriksaan terakhir yang
penting: kalau angka data uji jauh lebih buruk, berarti ada kebocoran di suatu
tempat pada proses pemilihan model.

### Menerjemahkan angka menjadi kalimat

`R²` sekitar 0,84 berarti model menjelaskan sekitar 84% keragaman harga.
`MAE` sekitar 0,30 berarti rata-rata salah tebak sekitar 30.000 USD.

Apakah itu bagus? **Tergantung pemakaiannya.** Untuk memperkirakan harga wilayah
secara kasar, memadai. Untuk menentukan harga jual sebuah rumah, sama sekali
tidak — datanya pun bukan tentang rumah tunggal.

Kalimat "modelnya bagus" tanpa menyebut untuk apa tidak berarti apa-apa. Selalu
kaitkan angka dengan keputusan yang akan diambil berdasarkan angka itu.

In [ ]:
# Grafik akhir pada data uji
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].scatter(y_uji, tebakan_uji, alpha=0.2, s=8, color="#4C72B0")
ax[0].plot([0, 5.2], [0, 5.2], color="#C44E52", linestyle="--", linewidth=2)
ax[0].set_xlabel("kenyataan")
ax[0].set_ylabel("tebakan")
ax[0].set_title(f"Data uji: prediksi vs kenyataan (R2 = {r2_uji:.3f})")

sisa_uji = y_uji - tebakan_uji
ax[1].scatter(tebakan_uji, sisa_uji, alpha=0.2, s=8, color="#55A868")
ax[1].axhline(0, color="#C44E52", linestyle="--", linewidth=2)
ax[1].set_xlabel("tebakan")
ax[1].set_ylabel("residual")
ax[1].set_title("Data uji: residual")

plt.tight_layout()
plt.show()

In [ ]:
# Kesalahan per rentang harga — lebih berguna daripada satu angka MAE
periksa_uji = pd.DataFrame({"kenyataan": y_uji.values, "tebakan": tebakan_uji})
periksa_uji["kelompok"] = pd.cut(periksa_uji["kenyataan"],
                                 bins=[0, 1, 2, 3, 4, 5.1],
                                 labels=["<1", "1-2", "2-3", "3-4", ">4"])

periksa_uji["salah_mutlak"] = (periksa_uji["kenyataan"] - periksa_uji["tebakan"]).abs()

per_kelompok = periksa_uji.groupby("kelompok", observed=True).agg(
    jumlah=("kenyataan", "size"),
    MAE=("salah_mutlak", "mean"),
    rata_kenyataan=("kenyataan", "mean"),
    rata_tebakan=("tebakan", "mean"),
)

print("MAE per rentang harga (ratusan ribu USD):")
print(per_kelompok.round(3).to_string())

Tabel ini jauh lebih informatif daripada MAE tunggal, dan memperlihatkan dua
hal yang tidak mungkin terlihat dari satu angka.

**Pertama, kesalahan naik seiring harga.** MAE kelompok termurah sekitar 0,19;
kelompok termahal sekitar tiga kali lebih besar. Jadi MAE keseluruhan sekitar
0,29 itu bukan gambaran yang berlaku merata — ia terlalu optimistis untuk blok
mahal dan terlalu pesimistis untuk blok murah.

**Kedua, ada kecenderungan sistematis.** Bandingkan kolom `rata_kenyataan` dengan
`rata_tebakan`: pada kelompok termurah model menebak **terlalu tinggi**, pada
kelompok termahal **terlalu rendah**. Model cenderung menarik tebakannya ke
tengah. Itu perilaku khas, dan pada kelompok `>4` diperkuat oleh sensor di 5,00
yang sudah kita temukan di Bagian 1.

Pemeriksaan seperti ini adalah bagian inti dari Responsible AI, bukan tambahan.
Model yang "rata-rata akurat" bisa jadi buruk, atau berbias satu arah, pada
kelompok tertentu — dan kalau kelompok itu adalah orang, bukan blok sensus, akibatnya
serius. Sebuah model penyaringan bantuan sosial yang akurat secara keseluruhan
tapi selalu salah pada kelompok termiskin adalah model yang gagal justru pada
orang yang paling membutuhkannya.

**Selalu pecah metrikmu per kelompok.** Satu angka menyembunyikan terlalu banyak.

---

## Bagian 11 — Menyimpan dan memakai model

In [ ]:
joblib.dump(model_akhir, "model_harga_rumah.joblib")
print("Model disimpan ke model_harga_rumah.joblib")

model_dimuat = joblib.load("model_harga_rumah.joblib")

# Data baru dalam bentuk ASLI (8 kolom). Fitur rekayasa ditambahkan
# dengan fungsi yang sama seperti saat pelatihan.
blok_baru = pd.DataFrame([
    # Blok berpendapatan tinggi, dekat San Francisco
    {"MedInc": 8.3, "HouseAge": 15, "AveRooms": 6.9, "AveBedrms": 1.0,
     "Population": 1200, "AveOccup": 2.8, "Latitude": 37.80, "Longitude": -122.40},
    # Blok berpendapatan rendah, pedalaman jauh dari kota besar
    {"MedInc": 1.8, "HouseAge": 40, "AveRooms": 4.2, "AveBedrms": 1.1,
     "Population": 900, "AveOccup": 3.6, "Latitude": 36.30, "Longitude": -119.80},
    # Blok menengah, pinggiran Los Angeles
    {"MedInc": 4.5, "HouseAge": 28, "AveRooms": 5.5, "AveBedrms": 1.05,
     "Population": 1600, "AveOccup": 3.0, "Latitude": 34.10, "Longitude": -118.30},
])

tebakan_baru = model_dimuat.predict(tambah_fitur(blok_baru))

print()
for i, t in enumerate(tebakan_baru):
    print(f"Blok {i + 1}: {t:.3f} (= {t * 100_000:,.0f} USD) | "
          f"MedInc {blok_baru.loc[i, 'MedInc']} | "
          f"jarak kota {tambah_fitur(blok_baru).loc[i, 'jarak_kota']:.2f}")

Ketiga tebakan itu masuk akal dan berurutan sesuai dugaan. Pemeriksaan semacam
ini — memasukkan beberapa kasus yang kamu sudah tahu jawabannya secara kasar —
adalah cara tercepat menemukan kesalahan pada pipeline. Lakukan ini sebelum
mendemokan apa pun.

### Menyatakan ketidakpastian pada regresi

Pada klasifikasi ada `predict_proba`. Pada regresi, tidak ada yang sepadan secara
langsung — tetapi kita bisa memakai **MAE per rentang** dari Bagian 10 sebagai
perkiraan ketidakpastian, dan menyampaikannya sebagai rentang alih-alih satu
angka.

Melaporkan satu angka tanpa rentang membuat model terkesan lebih pasti daripada
kenyataannya. Untuk aplikasi yang dipakai orang, itu masalah Responsible AI.

In [ ]:
print("Tebakan disampaikan sebagai rentang, memakai MAE sebagai perkiraan kesalahan:")
print()
for i, t in enumerate(tebakan_baru):
    bawah, atas = (t - mae_uji) * 100_000, (t + mae_uji) * 100_000
    print(f"Blok {i + 1}: sekitar {t * 100_000:,.0f} USD "
          f"(perkiraan {bawah:,.0f} - {atas:,.0f} USD)")

print()
print("Bandingkan dengan menyebut satu angka saja. Bentuk rentang lebih jujur,")
print("dan lebih mudah dipertanggungjawabkan saat ditanya.")

---

## Bagian 12 — Batasan dan Responsible AI

| # | Risiko / batasan | Mengapa nyata | Mitigasi |
|---|---|---|---|
| 1 | **Data dari sensus 1990** | harga rumah California sudah berubah drastis; model ini tidak berlaku untuk hari ini | nyatakan tahun datanya di setiap laporan; jangan dipakai untuk keputusan nyata |
| 2 | **Target tersensor di 500 ribu USD** | 4,8% data dipotong; model tidak bisa menebak di atas itu | batasan dinyatakan; kesalahan dilaporkan terpisah untuk kelompok `>4` |
| 3 | **Satu baris = satu blok, bukan satu rumah** | memakainya untuk menilai rumah tunggal adalah kesalahan penafsiran | dinyatakan pada kamus data dan pada presentasi |
| 4 | **Kesalahan tidak merata antar rentang harga** | MAE blok termahal sekitar 3x MAE blok termurah; blok murah ditebak terlalu tinggi, blok mahal terlalu rendah | MAE dilaporkan per kelompok (Bagian 10), bukan hanya keseluruhan |
| 5 | **Lintang dan bujur adalah fitur terkuat** | letak dapat menjadi pengganti tidak langsung bagi komposisi penduduk | jangan dipakai untuk keputusan atas orang (kredit, asuransi, bantuan) |
| 6 | **Satu angka tanpa rentang** | pemakai menganggap tebakan lebih pasti daripada kenyataannya | disampaikan sebagai rentang (Bagian 11) |

### Risiko nomor 5 perlu dibaca dua kali

Model ini sangat bergantung pada lokasi. Di banyak tempat, lokasi berkorelasi
dengan suku, agama, atau tingkat ekonomi penduduk. Model yang memakai lokasi
untuk mengambil keputusan tentang **orang** — siapa yang dapat pinjaman, siapa
yang dapat bantuan — bisa memperlakukan orang secara berbeda berdasarkan tempat
tinggalnya, tanpa pernah menyebut suku atau agamanya secara langsung.

Itulah pola yang sama dengan kasus Boston Housing di Bagian 1, hanya lebih
tersembunyi. Untuk menebak harga wilayah, lokasi wajar dipakai. Untuk memutuskan
sesuatu tentang orang, tidak.

**Cara memeriksanya pada proyekmu sendiri:** tanyakan "kalau keluaran model ini
salah, siapa yang dirugikan, dan apakah kerugiannya menimpa satu kelompok lebih
berat?" Kalau jawabannya ya, itu risiko yang wajib kamu tulis di proposal.

### Untuk proposal LKS-mu

Panduan LKS 2026 meminta minimal satu risiko beserta mitigasinya, dan penjelasan
di mana penilaian manusia berperan. Bentuk tabel di atas bisa langsung kamu pakai
— yang penting **spesifik pada datamu sendiri**, disertai cara mengukurnya, dan
disertai apa yang kamu lakukan sebagai akibatnya.

---

## Bagian 13 — Cek pemahaman

In [ ]:
# ---- Kerjakan ----

# a) Berapa baris data latih dan data uji?
n_latih, n_uji = None, None

# b) Berapa banyak baris pada SELURUH df yang targetnya tersensor (>= 5.0)?
n_tersensor = None

# c) Hitung MAE model_akhir pada data uji untuk blok dengan kenyataan < 2.0 saja.
#    Petunjuk: pakai penyaringan boolean atas y_uji dan tebakan_uji.
mae_murah = None

# d) Tiga fitur paling berpengaruh menurut permutation importance.
#    Isi sebagai himpunan (set) berisi tiga nama kolom; urutannya tidak penting.
tiga_terkuat = None


# ---- Pemeriksaan otomatis ----
assert (n_latih, n_uji) == (16512, 4128), f"(a) seharusnya (16512, 4128), dapatnya {(n_latih, n_uji)}"
assert n_tersensor == 992, f"(b) seharusnya 992, dapatnya {n_tersensor}"
assert mae_murah is not None and 0 < mae_murah < 1, \
    f"(c) MAE kelompok murah seharusnya antara 0 dan 1, dapatnya {mae_murah}"
assert tiga_terkuat == {"Latitude", "Longitude", "MedInc"}, \
    f"(d) tiga fitur terkuat belum tepat, dapatnya {tiga_terkuat}"

print("Semua pemeriksaan lolos.")
print(f"MAE kelompok termurah: {mae_murah:.4f} vs MAE keseluruhan: {mae_uji:.4f}")

---

## Bagian 14 — Latihan mandiri: regresi pada data sekolah sendiri

Kerjakan ulang seluruh alur notebook ini memakai data konsumsi listrik sekolah
pada paket materi: `03-praktikum/pengayaan-python/konsumsi-listrik-sekolah-sintetis.csv`.

Kasusnya: **menebak `kwh` dari `jam_pakai`, `jenis_ruang`, dan `hari`.** Ini
regresi, dan temanya sejalan dengan **Studi Kasus 2 (Lingkungan)** pada panduan
LKS 2026.

Langkahnya:

1. Unggah berkasnya ke Colab (Notebook 01 Bagian 8), lalu
   `df = pd.read_csv("konsumsi-listrik-sekolah-sintetis.csv")`.
2. EDA: delapan pertanyaan di Notebook 02 Bagian 4. Lihat sebaran `kwh` dan
   hubungannya dengan `jam_pakai`.
3. `y = df["kwh"]`, `X = df[["jam_pakai", "jenis_ruang", "hari"]]`.
4. **Perhatian pembagian data.** Setiap ruang muncul pada banyak tanggal. Kalau
   dibagi acak biasa, ruang yang sama bisa muncul di data latih dan data uji
   sekaligus — model "mengenali" ruangnya, bukan belajar polanya. Bagi
   **berdasarkan tanggal**: tanggal terakhir untuk uji, sisanya untuk latih.
   Ini bentuk kebocoran data yang paling sering terlewat pada data berwaktu, dan
   `train_test_split` biasa tidak melindungimu darinya.
5. `jenis_ruang` dan `hari` bertipe teks, jadi perlu `ColumnTransformer` dengan
   `OneHotEncoder` — lihat Notebook 03 Bagian 4.
6. Baseline `DummyRegressor`, lalu bandingkan minimal tiga model.
7. Laporkan MAE dalam satuan kWh, sebutkan jumlah data ujinya, dan buat grafik
   residual.
8. Tulis tabel risiko Responsible AI. Risiko pertama yang wajib disebutkan:
   **datanya sintetis, bukan pengukuran meteran sungguhan.**

**Latihan tambahan.** Setelah selesai, buat satu fitur rekayasa sendiri
(misalnya `kwh_per_jam` dari data latih, atau penanda akhir pekan), ukur apakah
ia memperbaiki RMSE, lalu bandingkan perbaikannya dengan simpangan
cross-validation. Kalau lebih kecil dari simpangan, perbaikannya belum terbukti.

---

## Ringkasan

### Klasifikasi dan regresi bersebelahan

| | Klasifikasi | Regresi |
|---|---|---|
| Kelas baseline | `DummyClassifier(strategy="most_frequent")` | `DummyRegressor(strategy="mean")` |
| Metrik utama | accuracy, F1 macro | MAE, RMSE, R² |
| `scoring` di CV | `"accuracy"`, `"f1_macro"` | `"neg_root_mean_squared_error"`, `"r2"` |
| Alat diagnosis | confusion matrix | plot residual, prediksi vs kenyataan |
| Ketidakpastian | `predict_proba` | rentang berdasarkan MAE |
| `stratify` saat split | ya | biasanya tidak |
| Metrik per kelompok | per kelas (`classification_report`) | per rentang nilai (`pd.cut` + `groupby`) |

### Yang berlaku pada keduanya

1. Bagi data sebelum melakukan apa pun.
2. Selalu punya baseline, dan sebutkan angkanya.
3. Prapemrosesan di dalam `Pipeline`, bukan manual.
4. Nilai dengan cross-validation di data latih; data uji dibuka sekali di akhir.
5. Sebutkan jumlah data setiap kali melaporkan metrik.
6. Pecah metrik per kelompok — satu angka menyembunyikan terlalu banyak.
7. Fitur yang lebih baik mengalahkan penyetelan parameter.
8. Tulis batasan dan risikonya. Itu bagian dari pekerjaannya, bukan pelengkap.

### Sudah selesai keempat notebook. Apa berikutnya

Untuk lomba, alur di notebook 03 dan 04 sudah lebih dari cukup untuk membangun
bagian AI sebuah aplikasi. Yang biasanya masih perlu dipelajari:

- **Kalau datanya gambar** (klasifikasi sampah) — Teachable Machine untuk
  prototipe cepat tanpa kode, atau *transfer learning* dengan Keras kalau ingin
  memakai Python. Lihat modul praktikum PilahPintar pada paket materi ini.
- **Kalau datanya teks** (pemeriksa hoaks, chatbot layanan) — `TfidfVectorizer`
  + `LogisticRegression` sebagai dasar; API model bahasa untuk yang lebih maju.
- **Kalau perlu jadi aplikasi** — Streamlit atau Gradio bisa mengubah model
  `.joblib` menjadi aplikasi web dalam 20–30 baris kode.

Satu nasihat terakhir yang sesuai dengan kriteria penilaian LKS 2026: panduan
lomba menyebut penilaian menekankan kreativitas, pemanfaatan AI yang tepat guna,
dan dampak nyata — **bukan kesempurnaan teknis**. Model sederhana yang kamu
pahami sepenuhnya, dengan batasan yang jujur dan peran manusia yang jelas, lebih
bernilai daripada model rumit yang tidak bisa kamu jelaskan saat ditanya.